# 10 Transformer le score en classes (régression → classification)

Dans le notebook 02, nous avons construit une cible continue :

$$\text{score\_compromis} = \sqrt{R_{\mathrm{UTS}} \times R_{\mathrm{allongement}}}$$

Les notebooks précédents ont traité ce score comme un problème de **régression**. Ici, nous le **découpons en classes ordonnées** (par exemple *Faible / Moyen / Élevé*) pour pouvoir aussi utiliser des **classifieurs** (régression logistique, arbres, SVM, etc.).

Découper un score continu est un choix de modélisation, pas une vérité physique. Nous cherchons donc des seuils que nous pouvons **justifier**, et nous gardons la même rigueur que dans les notebooks 02 et 03 : **tout ce qui est appris sur les données (ici, les seuils) l'est sur l'apprentissage uniquement**.

**Plan**
1. Charger le score et la partition développement / test.
2. Regarder la distribution du score : existe-t-il une coupure naturelle ?
3. Choisir une méthode de découpe et comparer 2, 3 et 4 classes.
4. Définir les classes et vérifier leur sens physique (UTS, allongement).
5. Les appliquer au test et dans chaque pli de validation croisée, sans fuite d'information.
6. Mesurer l'effet des frontières, puis vérifier rapidement que les classes sont apprenables.
7. Exporter le résultat pour les autres notebooks.

## 1 Charger les données et les fonctions communes

Nous repartons de `outputs/02/dataset_traction.csv` : ce fichier contient les entrées `X`, UTS, l'allongement, le score, le groupe de chaque ligne et la partition (développement / test). **Nous ne refaisons pas la séparation** : le test reste exactement celui du notebook 02.

Nous réutilisons les fonctions de `src/preparation.py` (`reference_rangs`, `poids_groupes`, `calculer_cible`, `creer_pipeline`) au lieu de les recopier, pour que tous les notebooks calculent la même chose.

> Prérequis : les fichiers `outputs/02/dataset_traction.csv` et `src/preparation.py` doivent être présents (branche de Marie ou `main` après fusion). Sinon, exécuter d'abord le notebook 02.

In [1]:
from pathlib import Path
import json
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.mixture import GaussianMixture
from sklearn.model_selection import GroupKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score,
                             cohen_kappa_score, confusion_matrix, ConfusionMatrixDisplay)

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
OUT = ROOT / 'outputs/10'
OUT.mkdir(parents=True, exist_ok=True)

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.preparation import poids_groupes, reference_rangs, calculer_cible, creer_pipeline

SEED = 42

ModuleNotFoundError: No module named 'src'

Nous chargeons le fichier en gardant les colonnes de texte comme du texte (comme dans les notebooks 02 et 03), puis nous séparons développement et test avec la colonne `partition`.

`variables_X` contient toutes les entrées : nous retirons les identifiants, UTS, l'allongement et le score, qui servent à construire la cible et ne doivent **jamais** être des entrées du modèle.

In [ ]:
categories = ['AC_DC', 'Electrode_pol', 'WeldType']
donnees = pd.read_csv(
    ROOT / 'outputs/02/dataset_traction.csv', index_col='row_id',
    dtype={c: str for c in categories + ['WeldID', 'partition']},
    keep_default_na=False, na_values=[''], float_precision='round_trip'
)

mesures = ['UTS_MPa', 'Elongation_pct']
colonnes_hors_X = ['groupe', 'partition', 'WeldID', 'score_compromis'] + mesures
variables_X = [c for c in donnees.columns if c not in colonnes_hors_X]

est_dev = donnees['partition'] == 'developpement'
est_test = donnees['partition'] == 'test'
assert (est_dev | est_test).all()

score_dev = donnees.loc[est_dev, 'score_compromis']
groupes_dev = donnees.loc[est_dev, 'groupe']
Y_dev = donnees.loc[est_dev, mesures]
assert set(groupes_dev).isdisjoint(set(donnees.loc[est_test, 'groupe']))

print(f'Développement : {est_dev.sum()} lignes, {groupes_dev.nunique()} groupes.')
print(f'Test réservé : {est_test.sum()} lignes.')
print(f'{len(variables_X)} variables d\'entrée avant sélection des colonnes.')

## 2 Regarder la distribution du score (développement uniquement)

Avant de choisir des seuils, nous regardons **comment le score est réparti**. Nous n'utilisons que le développement : le test ne doit pas influencer la définition des classes.

Ce qui nous intéresse : le score est-il en une seule cloche, ou en plusieurs bosses ? S'il y avait des bosses séparées par un creux, on pourrait placer les seuils dans les creux. Sans creux, tout seuil est une convention.

In [ ]:
print(score_dev.describe().round(3).to_string())
print(f"\nAsymétrie (skewness) : {score_dev.skew():.2f}")

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.hist(score_dev, bins=30, color='tab:blue', alpha=0.75, edgecolor='white')
ax.axvline(score_dev.mean(), color='black', linestyle='--', label=f'Moyenne = {score_dev.mean():.2f}')
ax.axvline(score_dev.median(), color='tab:red', linestyle=':', label=f'Médiane = {score_dev.median():.2f}')
ax.set_xlabel('score_compromis')
ax.set_ylabel('Nombre de lignes')
ax.set_title('Distribution du score sur le développement')
ax.legend()
fig.tight_layout()
fig.savefig(OUT / 'distribution_score.png', dpi=150)
plt.show()

La distribution est **en une seule cloche**, quasi symétrique (moyenne ≈ médiane, asymétrie proche de 0, légèrement négative : la queue est un peu plus longue du côté des faibles scores). Environ 70 % des lignes ont un score entre 0,3 et 0,6, et les scores au-dessus de 0,6 sont rares (≈ 7 %) : très peu de soudures ont à la fois une résistance et une ductilité élevées, ce qui est cohérent avec le compromis résistance/ductilité.

Cette allure n'est pas une surprise : le score est la moyenne géométrique de deux rangs, ce qui tasse en partie les valeurs vers le centre.

## 3 Existe-t-il des sous-populations naturelles ?

L'histogramme peut tromper : deux bosses proches peuvent se fondre en une seule cloche. Nous testons donc l'idée de façon plus objective avec un **mélange de gaussiennes (GMM)** ajusté sur le score en 1 à 4 composantes.

Nous comparons les modèles avec le **BIC** : plus il est bas, meilleur est le compromis entre qualité d'ajustement et complexité. Si 2 ou 3 composantes avaient un BIC nettement plus bas que 1, le score contiendrait des groupes naturels et nous pourrions nous appuyer dessus pour placer les seuils.

In [ ]:
valeurs = score_dev.to_numpy().reshape(-1, 1)
lignes = []
for k in range(1, 5):
    gmm = GaussianMixture(n_components=k, n_init=5, random_state=SEED).fit(valeurs)
    lignes.append({'composantes': k, 'BIC': gmm.bic(valeurs),
                   'moyennes': np.round(np.sort(gmm.means_.ravel()), 3).tolist()})
bic = pd.DataFrame(lignes)
display(bic.round(1))
print(f"Meilleur BIC (le plus bas) : {bic.loc[bic['BIC'].idxmin(), 'composantes']} composante(s).")

Le BIC est le plus bas avec **une seule composante** : rien dans les données ne justifie de séparer le score en groupes naturels.

**Conséquence importante.** Aucun creux ne nous indique où couper. Les seuils sont donc une **convention de projet**, comme l'était le choix des poids égaux dans le notebook 02. Il faut le dire clairement dans le rapport. En revanche, nous pouvons choisir une convention **simple, reproductible et équilibrée** : les **quantiles**.

## 4 Choisir la méthode : des quantiles pondérés par groupe

Découper en quantiles signifie : *« les 33 % de soudures les plus faibles → classe Faible, etc. »*. Avantages :
- les classes sont **équilibrées**, ce qui évite qu'un classifieur gagne en prédisant toujours la classe majoritaire ;
- la méthode est simple à expliquer et à refaire dans chaque pli de validation ;
- elle ne suppose aucune forme particulière de distribution (contrairement à « moyenne ± écart-type »).

**Point d'attention : les groupes.** Dans le notebook 02, chaque groupe de lignes (même dépôt présumé) pèse **1 au total** dans le calcul des rangs. Nous gardons la même règle pour les seuils : sinon un dépôt avec 4 essais compterait 4 fois plus qu'un dépôt avec 1 essai.

Astuce d'implémentation : `reference_rangs` donne déjà, pour chaque valeur du score, son **rang pondéré** entre 0 et 1. Le seuil du quantile `q` est simplement la valeur dont le rang vaut `q`, d'où une interpolation. Nous réutilisons ainsi exactement la même mécanique que pour la cible.

In [ ]:
def seuils_ponderes(score_apprentissage, groupes_apprentissage, quantiles):
    # Valeurs du score pour lesquelles le rang pondéré (un poids par groupe) vaut les quantiles demandés.
    valeurs_ref, rangs_ref = reference_rangs(score_apprentissage, groupes_apprentissage)
    return np.interp(quantiles, rangs_ref, valeurs_ref)


quantiles_test = [1/4, 1/3, 1/2, 2/3, 3/4]
comparaison = pd.DataFrame({
    'quantile': [f'{q:.3f}' for q in quantiles_test],
    'seuil_brut': [score_dev.quantile(q) for q in quantiles_test],
    'seuil_pondere_par_groupe': seuils_ponderes(score_dev, groupes_dev, quantiles_test),
})
comparaison['ecart'] = comparaison['seuil_pondere_par_groupe'] - comparaison['seuil_brut']
display(comparaison.round(3))

Les seuils pondérés sont légèrement plus bas que les seuils bruts (de 0,003 à 0,009) : les groupes à plusieurs essais ont plutôt des scores un peu plus élevés, et les pondérer à la baisse tire les quantiles vers le bas. L'écart est faible mais systématique. Nous gardons les **seuils pondérés** pour rester cohérents avec la façon dont le score lui-même a été construit.

## 5 Combien de classes ? Comparer 2, 3 et 4

Plus il y a de classes, plus l'information est fine, mais plus les classes sont **étroites** : beaucoup de soudures se retrouvent alors près d'un seuil, où la classe dépend d'une toute petite différence de score (donc très sensible au bruit de mesure).

Nous quantifions cela. Pour chaque découpage (2, 3, 4 classes équilibrées), nous calculons :
- les seuils obtenus ;
- l'effectif de la plus petite classe (pour vérifier qu'aucune classe n'est trop rare) ;
- la **part de lignes « proches d'un seuil »**, c'est-à-dire à moins de 0,03 d'un seuil (≈ 0,2 écart-type du score). Ce 0,03 est un ordre de grandeur arbitraire, mais il permet de comparer les découpages entre eux.

In [ ]:
MARGE = 0.03
lignes = []
for n_classes in [2, 3, 4]:
    quantiles = np.arange(1, n_classes) / n_classes
    seuils = seuils_ponderes(score_dev, groupes_dev, quantiles)
    classes = np.digitize(score_dev, seuils)
    distance = np.min(np.abs(score_dev.to_numpy()[:, None] - seuils[None, :]), axis=1)
    lignes.append({
        'n_classes': n_classes,
        'seuils': np.round(seuils, 3).tolist(),
        'effectif_min': int(np.bincount(classes).min()),
        'effectif_max': int(np.bincount(classes).max()),
        'part_proche_seuil_pct': 100 * (distance < MARGE).mean(),
    })
choix_classes = pd.DataFrame(lignes)
display(choix_classes.round(1))

Plus on ajoute de classes, plus la part de lignes proches d'une frontière augmente (les classes se rétrécissent) : environ **17 %** des lignes avec 2 classes, **29 %** avec 3 et **44 %** avec 4.

**Notre choix :**
- **3 classes (Faible / Moyen / Élevé)** comme découpage principal : c'est un bon compromis, avec une lecture intuitive (« mauvaise / moyenne / bonne soudure ») et des classes encore assez larges. C'est aussi ce que suggérait le notebook 08.
- **2 classes (sous / au-dessus de la médiane)** comme **variante binaire** : plus simple, avec la plus faible part de lignes proches du seuil (≈ 17 %), mais on perd la distinction entre « moyen » et « très bon ».
- **Nous écartons 4 classes** : près d'une ligne sur deux serait proche d'une frontière, ce qui rend les étiquettes très sensibles à l'incertitude des mesures, et l'effectif par classe diminue (≈ 135 lignes en moyenne en développement).

Ces choix restent des conventions : nous comparerons les deux versions plus loin.

## 6 Écrire la fonction qui définit les classes

Nous regroupons la méthode dans **une seule fonction** réutilisable, `definir_classes`. Elle prend :
- le score et les groupes de l'**apprentissage** (pour apprendre les seuils) ;
- le score à classer (apprentissage, validation ou test) ;
- les quantiles voulus.

Elle renvoie les classes (des entiers 0, 1, 2… **ordonnés**) et les seuils utilisés. Une ligne dont le score est exactement égal à un seuil va dans la classe supérieure (`score ≥ seuil`).

Nous gardons des entiers ordonnés plutôt que du texte car cela permet d'utiliser des métriques qui tiennent compte de l'ordre (voir section 10) et de comparer facilement les classes. Les noms lisibles sont dans un dictionnaire à part.

> Cette fonction pourra être déplacée dans `src/preparation.py` pour être partagée avec les autres notebooks.

In [ ]:
QUANTILES_3 = [1/3, 2/3]   # 3 classes : Faible / Moyen / Élevé
QUANTILES_2 = [1/2]        # variante binaire : sous / au-dessus de la médiane
NOMS_3 = {0: 'Faible', 1: 'Moyen', 2: 'Élevé'}
NOMS_2 = {0: 'Bas', 1: 'Haut'}


def definir_classes(score_apprentissage, groupes_apprentissage, score_a_classer, quantiles):
    # Les seuils sont appris sur l'apprentissage uniquement, puis appliqués à n'importe quel score.
    seuils = seuils_ponderes(score_apprentissage, groupes_apprentissage, quantiles)
    classes = pd.Series(np.digitize(score_a_classer, seuils), index=score_a_classer.index, name='classe')
    return classes, seuils


classes_dev_3, seuils_3 = definir_classes(score_dev, groupes_dev, score_dev, QUANTILES_3)
classes_dev_2, seuils_2 = definir_classes(score_dev, groupes_dev, score_dev, QUANTILES_2)
print(f'Seuils à 3 classes : {np.round(seuils_3, 3)}  (Faible < {seuils_3[0]:.3f} ≤ Moyen < {seuils_3[1]:.3f} ≤ Élevé)')
print(f'Seuil binaire      : {np.round(seuils_2, 3)}')

### Visualiser les seuils et les zones frontières

Nous superposons les seuils à l'histogramme. Les bandes grisées marquent la zone à ±0,03 autour de chaque seuil : une soudure dans cette bande aurait pu changer de classe avec une toute petite variation de mesure. Cette zone est une **limite intrinsèque** de toute discrétisation.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.8))
bornes = [score_dev.min(), *seuils_3, score_dev.max()]
couleurs = ['#d95f02', '#e6ab02', '#1b9e77']
for i, (bas, haut) in enumerate(zip(bornes[:-1], bornes[1:])):
    ax.axvspan(bas, haut, color=couleurs[i], alpha=0.12)
ax.hist(score_dev, bins=30, color='tab:blue', alpha=0.7, edgecolor='white')
for s in seuils_3:
    ax.axvline(s, color='black', linestyle='--')
    ax.axvspan(s - MARGE, s + MARGE, color='grey', alpha=0.25)
ymax = ax.get_ylim()[1]
ax.set_ylim(0, ymax * 1.12)
milieux = [(bornes[i] + bornes[i + 1]) / 2 for i in range(3)]
for m, nom in zip(milieux, NOMS_3.values()):
    ax.text(m, ymax * 1.10, nom, ha='center', va='top', fontsize=12, fontweight='bold')
ax.set_xlabel('score_compromis')
ax.set_ylabel('Nombre de lignes')
ax.set_title('Découpage en 3 classes (bandes grises : ±0,03 autour des seuils)')
fig.tight_layout()
fig.savefig(OUT / 'seuils_3_classes.png', dpi=150)
plt.show()

## 7 Vérifier l'équilibre des classes

Nous comptons les lignes de chaque classe, **de deux façons** : le nombre brut de lignes, et la part du **poids** (un poids par groupe). Les deux doivent être proches de 1/3 : les seuils ont été construits sur le poids, donc le poids est équilibré par construction, tandis que le nombre brut de lignes peut s'en écarter un peu.

Des classes équilibrées sont utiles : une simple prédiction de la classe majoritaire ne donnera qu'environ 33 % de bonnes réponses, ce qui fournit une référence claire à battre.

In [ ]:
poids_dev = poids_groupes(groupes_dev)
effectifs = pd.DataFrame({
    'classe': [NOMS_3[c] for c in sorted(NOMS_3)],
    'lignes': np.bincount(classes_dev_3, minlength=3),
    'part_lignes_pct': 100 * np.bincount(classes_dev_3, minlength=3) / len(classes_dev_3),
    'part_poids_pct': 100 * np.bincount(classes_dev_3, weights=poids_dev, minlength=3) / poids_dev.sum(),
}).set_index('classe')
display(effectifs.round(1))

fig, ax = plt.subplots(figsize=(6, 3.6))
ax.bar(effectifs.index, effectifs['lignes'], color=couleurs)
for i, v in enumerate(effectifs['lignes']):
    ax.text(i, v + 2, str(v), ha='center')
ax.set_ylabel('Lignes (développement)')
ax.set_title('Effectifs des 3 classes')
fig.tight_layout()
fig.savefig(OUT / 'effectifs_classes.png', dpi=150)
plt.show()

Les trois classes sont équilibrées (≈ 33 % chacune). Comme attendu, la part du **poids** est quasiment exactement 1/3, tandis que le nombre brut de lignes s'en écarte légèrement (la classe Élevé a un peu plus de lignes : ses groupes comptent en moyenne plus d'essais).

## 8 Que signifient ces classes en unités physiques ?

Le score est abstrait (un rang combiné). Pour interpréter les classes, nous revenons aux **mesures d'origine** : nous affichons chaque soudure dans le plan (UTS, allongement), colorée par classe, et nous calculons les médianes par classe.

Nous regardons aussi la corrélation entre UTS et allongement. Dans les aciers, ces deux propriétés sont généralement en **opposition** (un acier plus résistant est souvent moins ductile) : cela conditionne la lecture du score.

In [ ]:
tableau = Y_dev.assign(classe=classes_dev_3.map(NOMS_3), score=score_dev)
mediane_par_classe = tableau.groupby('classe')[['UTS_MPa', 'Elongation_pct', 'score']].median().loc[list(NOMS_3.values())]
display(mediane_par_classe.round(2))
print('Corrélation de Spearman UTS / allongement :',
      round(Y_dev['UTS_MPa'].corr(Y_dev['Elongation_pct'], method='spearman'), 2))

fig, ax = plt.subplots(figsize=(8, 5.5))
for c, nom in NOMS_3.items():
    masque = classes_dev_3 == c
    ax.scatter(Y_dev.loc[masque, 'UTS_MPa'], Y_dev.loc[masque, 'Elongation_pct'],
               s=22, alpha=0.7, color=couleurs[c], label=nom)
ax.set_xlabel('UTS (MPa)')
ax.set_ylabel('Allongement (%)')
ax.set_title('Les 3 classes dans le plan résistance / ductilité')
ax.legend(title='Classe')
fig.tight_layout()
fig.savefig(OUT / 'classes_uts_allongement.png', dpi=150)
plt.show()

**Lecture.** UTS et allongement sont fortement anti-corrélés (ρ ≈ −0,8) : le nuage de points suit une diagonale descendante. Les classes sont des **bandes parallèles à cette diagonale** :
- la classe **Faible** apparaît dans **deux zones opposées** : en haut à gauche (faible résistance, mais forte ductilité) et en bas à droite (forte résistance, mais faible ductilité). Ce sont deux façons d'avoir un mauvais compromis : une propriété est bonne, l'autre est trop basse ;
- la classe **Élevé** forme la bande supérieure du nuage : ce sont les soudures qui, pour une résistance donnée, **conservent plus de ductilité que les autres**. L'UTS médiane augmente avec la classe, alors que l'allongement médian reste du même ordre de grandeur ;
- la classe **Moyen** est la bande intermédiaire.

C'est exactement le comportement voulu pour un « score de compromis » : ce n'est pas simplement « plus d'UTS », c'est un équilibre. Il faut aussi retenir que les trois classes **se chevauchent** dans le plan (UTS, allongement) : un modèle aura du mal à séparer parfaitement les classes voisines, ce que nous verrons à la section 10.

## 9 Appliquer les classes sans fuite d'information

### 9.1 Sur le test

Nous classons les lignes du test **avec les seuils appris sur le développement**, sans jamais recalculer de quantiles sur le test. Si nous les recalculions, nous « tricherions » : les classes du test seraient définies avec l'information du test, et ses proportions seraient artificiellement équilibrées.

Le test étant petit (127 lignes) et choisi par groupes, ses proportions de classes peuvent donc s'écarter de 1/3. C'est normal et même sain : c'est la vraie distribution que le modèle rencontrera.

In [ ]:
score_test = donnees.loc[est_test, 'score_compromis']
classes_test_3, _ = definir_classes(score_dev, groupes_dev, score_test, QUANTILES_3)   # seuils du développement

repartition = pd.DataFrame({
    'développement_%': 100 * classes_dev_3.value_counts(normalize=True).sort_index(),
    'test_%': 100 * classes_test_3.value_counts(normalize=True).sort_index(),
    'test_lignes': classes_test_3.value_counts().sort_index(),
})
repartition.index = [NOMS_3[i] for i in repartition.index]
display(repartition.round(1))

Le test est moins équilibré que le développement : la classe Faible y est sous-représentée (≈ 24 % contre 32 %) et la classe Élevé sur-représentée (≈ 40 % contre 35 %). Ce n'est pas un défaut : avec 127 lignes, quelques groupes suffisent à déplacer les proportions d'une dizaine de points. Nous gardons ces classes telles quelles, et nous utiliserons des métriques qui ne se laissent pas piéger par un léger déséquilibre (voir section 10).

### 9.2 Dans chaque pli de validation croisée

C'est ici que la rigueur est la plus importante. Pour comparer des modèles en validation croisée, **dans chaque pli** :
1. le score est **recalculé** à partir de la référence de rangs du pli (fonction `calculer_cible`, comme dans le notebook 02) ;
2. les seuils sont **recalculés** sur l'apprentissage du pli ;
3. les classes de la validation sont obtenues avec ces seuils.

Sans cela, les seuils (donc les étiquettes de validation) utiliseraient des lignes de validation : c'est une **fuite d'information**, qui rendrait les résultats trop optimistes.

La fonction `plis_classes` ci-dessous suit la même logique que `plis_cibles` du notebook 02, avec une étape de plus. Les groupes restent séparés entre apprentissage et validation (`GroupKFold`).

In [ ]:
def plis_classes(quantiles, n_plis=5):
    # Générateur : pour chaque pli, renvoie les identifiants, les classes d'apprentissage et de validation, et les seuils.
    validation = GroupKFold(n_splits=n_plis)
    for train, val in validation.split(Y_dev, groups=groupes_dev):
        tr, va = Y_dev.index[train], Y_dev.index[val]
        assert set(groupes_dev.loc[tr]).isdisjoint(set(groupes_dev.loc[va]))
        score_tr = calculer_cible(Y_dev.loc[tr], groupes_dev.loc[tr], Y_dev.loc[tr])
        score_va = calculer_cible(Y_dev.loc[tr], groupes_dev.loc[tr], Y_dev.loc[va])
        classes_tr, seuils = definir_classes(score_tr, groupes_dev.loc[tr], score_tr, quantiles)
        classes_va = pd.Series(np.digitize(score_va, seuils), index=va, name='classe')
        yield tr, va, classes_tr, classes_va, seuils


lignes = []
for numero, (tr, va, c_tr, c_va, seuils) in enumerate(plis_classes(QUANTILES_3), start=1):
    ligne = {'pli': numero, 'seuil_1': seuils[0], 'seuil_2': seuils[1]}
    for c, nom in NOMS_3.items():
        ligne[f'validation_{nom}'] = int((c_va == c).sum())
    lignes.append(ligne)
stabilite = pd.DataFrame(lignes).set_index('pli')
display(stabilite.round(3))
print('Seuils sur le développement entier :', np.round(seuils_3, 3))

Les seuils varient peu d'un pli à l'autre (au plus 0,01 d'écart avec les seuils du développement entier : entre 0,344 et 0,358 pour le premier seuil, entre 0,471 et 0,477 pour le second). Cela montre que la découpe est **stable** : elle ne dépend pas d'un petit nombre de lignes. Les classes de validation sont, elles, moins équilibrées que celles d'apprentissage (de 26 à 47 lignes par classe selon les plis), car les groupes sont répartis entre les plis et chaque pli ne contient qu'une centaine de lignes.

**À retenir pour la suite : pour comparer des classifieurs en validation croisée, il faut utiliser `plis_classes`, et non la colonne de classes exportée à la fin de ce notebook.** Cette colonne est utile pour regarder les résultats ou pour le test final, comme `score_compromis` dans le notebook 02.

## 10 Les classes sont-elles apprenables ? Un contrôle rapide

Avant de transmettre ces classes à l'équipe, nous vérifions qu'elles ont un **sens pour un modèle** : un classifieur doit faire nettement mieux qu'un hasard bien informé. **Il ne s'agit pas de la comparaison finale des modèles** : nous utilisons une seule forêt aléatoire, sans réglage, uniquement comme thermomètre.

Protocole : validation croisée en 5 plis avec groupes séparés, classes recalculées dans chaque pli (section 9.2), préparation de `X` ajustée sur l'apprentissage du pli (`creer_pipeline`). La référence est un classifieur qui prédit toujours la classe majoritaire de l'apprentissage.

Un avertissement de scikit-learn (« unknown categories ») peut apparaître : une catégorie rare absente de l'apprentissage d'un pli est alors codée comme la catégorie de référence. C'est la limite déjà signalée dans le notebook 03, sans conséquence ici.

**Choix des métriques**, car la bonne métrique dépend de la nature des classes :
- **Exactitude (accuracy)** : simple, mais trompeuse si les classes sont déséquilibrées ;
- **Exactitude équilibrée** et **F1 macro** : chaque classe pèse autant, quelle que soit sa taille ;
- **Kappa quadratique pondéré** : tient compte de l'**ordre** des classes (se tromper de Faible à Élevé est plus grave que de Faible à Moyen). Il vaut 0 pour le hasard et 1 pour un accord parfait.

In [ ]:
def controle_classification(quantiles):
    vrais, predits, references = [], [], []
    for tr, va, c_tr, c_va, _ in plis_classes(quantiles):
        X_tr, X_va = donnees.loc[tr, variables_X], donnees.loc[va, variables_X]
        foret = creer_pipeline(X_tr, RandomForestClassifier(
            n_estimators=300, class_weight='balanced', random_state=SEED, n_jobs=-1))
        foret.fit(X_tr, c_tr)
        majoritaire = DummyClassifier(strategy='prior').fit(X_tr, c_tr)
        vrais += c_va.tolist()
        predits += foret.predict(X_va).tolist()
        references += majoritaire.predict(X_va).tolist()
    return np.array(vrais), np.array(predits), np.array(references)


def resume_metriques(vrais, predits):
    return {'exactitude': accuracy_score(vrais, predits),
            'exactitude_equilibree': balanced_accuracy_score(vrais, predits),
            'F1_macro': f1_score(vrais, predits, average='macro'),
            'kappa_quadratique': cohen_kappa_score(vrais, predits, weights='quadratic')}


resultats = {}
for nom, quantiles in [('3 classes', QUANTILES_3), ('2 classes', QUANTILES_2)]:
    vrais, predits, refs = controle_classification(quantiles)
    resultats[nom] = (vrais, predits)
    resultats_nom = {'forêt aléatoire': resume_metriques(vrais, predits),
                     'toujours la classe majoritaire': resume_metriques(vrais, refs)}
    print(nom)
    display(pd.DataFrame(resultats_nom).T.round(3))

**Lecture.**
- Dans les deux cas, la forêt est **nettement meilleure que la référence** (F1 macro de 0,60 contre 0,17 en 3 classes ; 0,73 contre 0,34 en 2 classes). Les classes contiennent donc un vrai signal prédictible à partir de la composition et du procédé : elles ne sont pas du bruit.
- La version à 2 classes obtient de meilleurs scores bruts, mais ce n'est pas une comparaison équitable : **une tâche plus facile n'est pas une meilleure tâche**. Elle répond à une question plus grossière, et ses métriques ne sont pas comparables à celles de la version à 3 classes (le kappa, par exemple, est calculé sur une autre échelle de classes).
- Le kappa quadratique à 3 classes (≈ 0,61) est nettement positif : le modèle n'est pas seulement exact, il respecte globalement l'ordre.
- Ces chiffres sont des **ordres de grandeur** : avec une autre graine aléatoire pour la forêt, le F1 macro varie d'environ ±0,02. Pas de conclusion fine à en tirer.

### Où sont les erreurs ?
Nous regardons la matrice de confusion (en 3 classes) : les erreurs se situent-elles entre classes voisines (effet de frontière) ou entre classes extrêmes (vrai défaut du modèle) ?

In [ ]:
vrais_3, predits_3 = resultats['3 classes']
matrice = confusion_matrix(vrais_3, predits_3)

fig, ax = plt.subplots(figsize=(5, 4.5))
ConfusionMatrixDisplay(matrice, display_labels=list(NOMS_3.values())).plot(ax=ax, cmap='Blues', colorbar=False)
ax.set_xlabel('Classe prédite')
ax.set_ylabel('Classe réelle')
ax.set_title('Matrice de confusion cumulée (5 plis)')
fig.tight_layout()
fig.savefig(OUT / 'matrice_confusion_3_classes.png', dpi=150)
plt.show()

erreurs = vrais_3 != predits_3
adjacentes = np.abs(vrais_3 - predits_3) == 1
print(f'Erreurs : {erreurs.sum()} sur {len(vrais_3)} lignes ({100 * erreurs.mean():.0f} %).')
print(f'Part des erreurs entre classes voisines : {100 * adjacentes.sum() / erreurs.sum():.0f} %.')
print(f'Erreurs Faible ↔ Élevé : {int((np.abs(vrais_3 - predits_3) == 2).sum())}.')

**Lecture.** La grande majorité des erreurs (près de 9 sur 10) se font entre **classes voisines**, et seulement 24 (sur 214 erreurs) entre Faible et Élevé. C'est exactement ce que prédit la section 5 : la classe Moyen est « prise en sandwich » et beaucoup de soudures se trouvent près des frontières. Ces erreurs sont en grande partie inévitables : une soudure à 0,34 est « Faible » et une autre à 0,36 est « Moyen », alors qu'elles sont presque identiques physiquement.

Cela a une conséquence pratique pour la comparaison des modèles : **le F1 macro seul masque cette structure ; le kappa quadratique et la matrice de confusion sont indispensables dans le rapport.**

## 11 Exporter les classes et les choix

Nous sauvegardons :
- `dataset_classes.csv` : le dataset du notebook 02 avec deux colonnes en plus, `classe_3` (Faible/Moyen/Élevé en 0/1/2) et `classe_2` (Bas/Haut en 0/1). Les seuils viennent du **développement** et sont appliqués à toutes les lignes, test compris ;
- `manifest.json` : les seuils, les quantiles et les noms de classes, pour pouvoir tout reproduire et le citer dans le rapport.

Rappel : cette colonne sert à lire les résultats ou à l'évaluation finale sur le test. **Pour la validation croisée, il faut recalculer les classes dans chaque pli avec `plis_classes`.**

In [ ]:
score_tous = donnees['score_compromis']
classes_tous_3, _ = definir_classes(score_dev, groupes_dev, score_tous, QUANTILES_3)
classes_tous_2, _ = definir_classes(score_dev, groupes_dev, score_tous, QUANTILES_2)

export = donnees.assign(classe_3=classes_tous_3, classe_2=classes_tous_2)
assert (export.loc[est_dev, 'classe_3'] == classes_dev_3).all()
export.to_csv(OUT / 'dataset_classes.csv', index_label='row_id')

manifest = {
    'seed': SEED,
    'methode': 'quantiles du score_compromis, ponderes par groupe (poids total 1 par groupe), appris sur le developpement',
    'regle': 'classe superieure si score >= seuil',
    'classes_3': {'quantiles': QUANTILES_3, 'seuils': seuils_3.tolist(), 'noms': NOMS_3},
    'classes_2': {'quantiles': QUANTILES_2, 'seuils': seuils_2.tolist(), 'noms': NOMS_2},
    'validation_croisee': "recalculer score puis seuils dans chaque pli avec plis_classes",
}
(OUT / 'manifest.json').write_text(json.dumps(manifest, ensure_ascii=False, indent=2))
print('Fichiers enregistrés dans', OUT)
display(export[['score_compromis', 'classe_3', 'classe_2']].head())

## Conclusion

**Ce que nous avons fait.** Nous avons transformé le score continu `score_compromis` en **3 classes ordonnées** (Faible / Moyen / Élevé), avec une **variante binaire** (Bas / Haut), pour pouvoir utiliser des classifieurs.

**Pourquoi ces choix.**
- Le score n'a **aucune coupure naturelle** (distribution unimodale, BIC minimal avec une seule composante) : les seuils sont une convention, assumée comme telle.
- Nous avons choisi des **quantiles pondérés par groupe** : classes équilibrées, méthode simple, cohérente avec le calcul des rangs du notebook 02 (un poids par groupe).
- Nous retenons **3 classes** : compromis entre lisibilité et largeur des classes. Nous écartons 4 classes, trop étroites. Le découpage en 2 classes reste disponible.
- **Aucune fuite d'information** : seuils appris sur le développement, appliqués tels quels au test, et recalculés dans chaque pli de validation (`plis_classes`).

**Ce que montrent les vérifications.**
- Les seuils sont **stables** d'un pli à l'autre.
- Les classes ont un **sens physique** : l'UTS médiane augmente avec la classe sans perte d'allongement, reflet du compromis recherché.
- Elles sont **apprenables** : une forêt non réglée dépasse très nettement la prédiction de la classe majoritaire.

**Limites à écrire dans le rapport.**
- Les seuils sont **relatifs au corpus** : « Élevé » signifie « parmi le tiers supérieur de ces soudures », pas « conforme à une norme industrielle ».
- Une part importante des soudures est **proche d'une frontière** (≈ 29 % à moins de 0,03 d'un seuil) : deux soudures presque identiques peuvent avoir des classes différentes. La plupart des erreurs se font donc entre classes voisines, et une partie est irréductible.
- Découper un score continu **fait perdre de l'information** (nous perdons la distance au seuil) ; nous gardons le score pour l'interpréter.
- Les poids égaux du score (notebook 02) influencent directement les classes : un autre poids changerait le classement, donc les étiquettes.

**Pour la suite.**
1. Comparer les classifieurs avec `plis_classes` et rapporter **F1 macro, exactitude équilibrée, kappa quadratique et matrice de confusion** (pas l'exactitude seule).
2. Comparer 2 et 3 classes dans le rapport en précisant que les deux tâches ne sont pas équivalentes.
3. Éventuellement, tester la sensibilité des classes à d'autres poids du score ou à d'autres quantiles (par exemple 25 / 50 / 25).
4. L'apprentissage semi-supervisé (notebook 09) pourra utiliser ces classes comme étiquettes.